# Autoencoders

Um **autoencoder** aprende a **comprimir** cada amostra em poucas coordenadas e a **reconstruí-la** a partir delas. A ideia é a mesma da reconstrução do PCA, mas a compressão é feita por uma **rede neural**, o que permite seguir formas curvas nos dados, como o UMAP. Não é preciso conhecer redes neurais: construímos a ideia do zero, olhamos o espaço onde ficam as amostras comprimidas e terminamos usando o autoencoder para remover ruído de imagens.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sklearn.datasets import fetch_openml
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## Dataset: MNIST

O MNIST tem 70 mil imagens de dígitos manuscritos de 28×28 pixels, ou seja, vetores de $D = 784$ dimensões. Dividimos os pixels por 255 para que fiquem entre 0 (preto) e 1 (branco) e usamos 10 mil imagens para treino e 2 mil para teste. Na primeira execução o Scikit-Learn baixa o dataset, com cerca de 50 MB.

In [ ]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X = X / 255
y = y.astype(int)

X_train, y_train = X[:10_000], y[:10_000]
X_test, y_test = X[60_000:62_000], y[60_000:62_000]

plt.figure(figsize=(15, 3.5))
for k in range(20):
    plt.subplot(2, 10, k + 1)
    plt.imshow(X_train[k].reshape(28, 28), cmap='gray')
    plt.title(y_train[k])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

## Fundamentação Matemática

### Comprimir e Reconstruir

O **codificador** $f$ transforma uma amostra $x \in \mathbb{R}^D$ num código $z = f(x)$ com apenas $m$ coordenadas, e o **decodificador** $g$ reconstrói a amostra a partir do código, $\hat{x} = g(z)$. O espaço dos códigos é o **espaço latente**. As duas partes são treinadas juntas para minimizar o erro de reconstrução

$$ \mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \| x_i - g(f(x_i)) \|^2 $$

O alvo de cada amostra é ela mesma, então o treino é não supervisionado. Como $m$ é bem menor que $D$, copiar a entrada é impossível, e a rede precisa descobrir quais características dos dados são essenciais. Os parâmetros de $f$ e $g$, os pesos, são ajustados por **descida de gradiente**, como as posições dos pontos no t-SNE e no UMAP.

### Redes Neurais

Uma **camada** de rede neural é uma transformação linear seguida de uma função aplicada a cada coordenada:

$$ h = \sigma(W x + b) $$

em que $W$ é uma matriz de pesos, $b$ é o **viés** e $\sigma$ é a **função de ativação**. Cada coordenada de $h$ é um **neurônio**, e uma rede é uma sequência de camadas. Num autoencoder, as camadas estreitam até o código, o **gargalo**, e depois alargam de volta até $D$.

A ativação mais usada é a **ReLU**, $\sigma(t) = \max(0, t)$, que zera os valores negativos. É ela que torna a rede não linear: sem ativação, o autoencoder é uma projeção linear e, no máximo, iguala o PCA.

## Implementação

O Scikit-Learn não tem uma classe de autoencoder, mas tem o `MLPRegressor`, uma rede neural para regressão. Treinando com `fit(X, X)`, o alvo é a própria entrada. O parâmetro `hidden_layer_sizes` lista o número de neurônios das camadas intermediárias e `max_iter` é o número de **épocas**, as passadas pelos dados de treino. Usamos camadas de 256, 16 e 256 neurônios, ou seja, um código de 16 dimensões. Com 50 épocas o treino não converge completamente, e desligamos o aviso correspondente nos imports.

In [ ]:
ae = MLPRegressor(hidden_layer_sizes=(256, 16, 256), activation='relu', max_iter=50, random_state=42)
ae.fit(X_train, X_train)

[W.shape for W in ae.coefs_]

São quatro matrizes de pesos em `coefs_`, com os vieses em `intercepts_`: as duas primeiras formam o codificador e as duas últimas, o decodificador. Com elas escrevemos as duas metades da rede à mão. Como as amostras estão nas linhas, a multiplicação fica `X @ W`, e na saída usamos `np.clip` para manter os pixels entre 0 e 1.

In [ ]:
def encode(model, X):
    H = np.maximum(0, X @ model.coefs_[0] + model.intercepts_[0])      # 784 -> 256
    return np.maximum(0, H @ model.coefs_[1] + model.intercepts_[1])   # 256 -> 16


def decode(model, Z):
    H = np.maximum(0, Z @ model.coefs_[2] + model.intercepts_[2])      # 16 -> 256
    return np.clip(H @ model.coefs_[3] + model.intercepts_[3], 0, 1)   # 256 -> 784


Z_test = encode(ae, X_test)
X_ae = decode(ae, Z_test)
print(f"Maior diferença para o predict da biblioteca: {np.abs(X_ae - np.clip(ae.predict(X_test), 0, 1)).max():.1e}")

As duas funções reproduzem o `predict` da biblioteca: a rede é só multiplicações de matrizes intercaladas com a ReLU. Vendo as reconstruções:

In [ ]:
def plot_rows(rows):
    # cada linha da figura mostra 10 imagens, com o rótulo à esquerda
    plt.figure(figsize=(15, 1.6 * len(rows)))
    for r, (images, label) in enumerate(rows):
        for c in range(10):
            plt.subplot(len(rows), 10, r * 10 + c + 1)
            plt.imshow(images[c].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
            plt.xticks([])
            plt.yticks([])
            if c == 0:
                plt.ylabel(label, fontsize=11)
    plt.tight_layout()
    plt.show()


plot_rows([(X_test, 'original'), (X_ae, 'autoencoder')])

A partir de apenas 16 números, a maioria dos dígitos é reconstruída de forma reconhecível, com traços mais borrados. Os erros aparecem nos exemplos de traçado incomum.

### O Espaço Latente

Para enxergar os códigos de 16 dimensões, projetamos em 2D com o UMAP e colorimos pelo dígito, que o autoencoder nunca viu.

In [ ]:
Z_2d = umap.UMAP(random_state=42).fit_transform(Z_test)

plt.figure(figsize=(8, 6.5))
plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_test, cmap='tab10', s=6, alpha=0.7)
for digit in range(10):
    # escreve o dígito no centro do seu grupo
    center = np.median(Z_2d[y_test == digit], axis=0)
    plt.text(*center, str(digit), fontsize=16, weight='bold', ha='center', va='center',
             bbox=dict(boxstyle='circle', facecolor='white', alpha=0.8))
plt.xticks([])
plt.yticks([])
plt.title('Códigos de 16 dimensões, projetados em 2D')
plt.show()

Os dígitos formam grupos sem que nenhum rótulo tenha sido usado: reconstruir bem imagens parecidas exige códigos parecidos. Dígitos de traçado semelhante, como 4, 7 e 9 ou 3, 5 e 8, ficam próximos e se misturam.

Também podemos andar em linha reta entre os códigos de duas imagens, $z(t) = (1-t)\,z_1 + t\,z_2$, e decodificar cada ponto do caminho.

In [ ]:
def interpolate(model, x1, x2, steps=10):
    z1, z2 = encode(model, np.array([x1, x2]))
    t = np.linspace(0, 1, steps)[:, None]
    return decode(model, (1 - t) * z1 + t * z2)


i = 0
same = np.where(y_test == y_test[i])[0][1]
other = np.where(y_test != y_test[i])[0][0]
plot_rows([(interpolate(ae, X_test[i], X_test[same]), f'{y_test[i]} para {y_test[same]}'),
           (interpolate(ae, X_test[i], X_test[other]), f'{y_test[i]} para {y_test[other]}')])

Entre dois exemplares do mesmo dígito, um traço se deforma continuamente no outro. Entre dígitos diferentes, os quadros do meio são formas ambíguas, porque nada no treino garante que os pontos entre dois códigos correspondam a imagens reais.

## Aplicação: Remoção de Ruído

O **denoising autoencoder** aprende a reconstruir a imagem limpa a partir de uma versão com ruído, o que no `MLPRegressor` é só `fit(X_noisy, X)`. Como copiar a entrada reproduziria o ruído, a rede precisa aprender como são as imagens válidas. Usamos ruído gaussiano com desvio padrão 0.4.

In [ ]:
rng = np.random.default_rng(0)


def add_noise(X, std):
    return np.clip(X + std * rng.normal(size=X.shape), 0, 1)


dae = MLPRegressor(hidden_layer_sizes=(256, 16, 256), activation='relu', max_iter=50, random_state=42)
dae.fit(add_noise(X_train, 0.4), X_train)

X_noisy = add_noise(X_test[:10], 0.4)
plot_rows([(X_test, 'original'),
           (X_noisy, 'com ruído'),
           (decode(ae, encode(ae, X_noisy)), 'autoencoder'),
           (decode(dae, encode(dae, X_noisy)), 'denoising')])

O autoencoder comum, que só viu imagens limpas, reconstrói manchas e, em vários casos, o dígito errado. O denoising recupera a maioria dos dígitos, com traços mais grossos. Medindo o erro em relação à imagem limpa:

In [ ]:
results = {}
for std in [0.0, 0.2, 0.4, 0.8]:
    X_corrupted = add_noise(X_test, std)
    results[std] = {'autoencoder': np.mean((X_test - decode(ae, encode(ae, X_corrupted))) ** 2),
                    'denoising': np.mean((X_test - decode(dae, encode(dae, X_corrupted))) ** 2)}

pd.DataFrame(results).T.rename_axis('ruído').round(5)

O autoencoder comum só ganha em imagens limpas, para as quais foi treinado; com qualquer nível de ruído, o denoising é melhor.

## Custo e Limitações

O resultado depende da inicialização, do número de épocas e da arquitetura, e duas sementes diferentes produzem espaços latentes diferentes. As coordenadas do código não têm ordem de importância nem significado próprio, e com a ReLU no gargalo alguns neurônios podem ficar sempre em zero. O espaço latente também não gera dados novos de forma confiável, como mostrou a interpolação, e resolver isso é a ideia dos **autoencoders variacionais**. Por fim, a rede tem centenas de milhares de pesos e precisa de muitos dados; na prática, autoencoders são treinados com bibliotecas como o PyTorch, que permitem camadas próprias para imagens e o uso de GPU.

## Exercícios

### Exercício 1: Gargalo de Duas Dimensões

Treine um autoencoder com `hidden_layer_sizes=(256, 64, 2, 64, 256)`, `activation='tanh'` e 100 épocas, e adapte `encode` e `decode` para as três camadas de cada metade, trocando `np.maximum(0, ...)` por `np.tanh(...)`. Plote os códigos de teste coloridos pelo dígito. Depois, monte uma grade regular de pontos $(z_1, z_2)$ cobrindo a região ocupada, decodifique cada ponto e mostre o resultado como um mosaico de imagens. Que regiões produzem dígitos reconhecíveis? Usamos `tanh` porque, com a ReLU e só dois neurônios no gargalo, é comum um deles ficar sempre em zero.

In [ ]:
# Seu código aqui

### Exercício 2: Tamanho do Gargalo

Treine autoencoders com `hidden_layer_sizes=(256, m, 256)` para $m$ igual a 2, 4, 8, 16 e 32, plote o erro de reconstrução no teste em função de $m$ e mostre as reconstruções de algumas imagens para cada $m$. A partir de que tamanho os dígitos ficam reconhecíveis? Dobrar $m$ sempre traz o mesmo ganho?

In [ ]:
# Seu código aqui

### Exercício 3: Detecção de Anomalias

Treine um autoencoder usando apenas as imagens do dígito 1 do conjunto de treino. Calcule o erro de reconstrução, imagem a imagem, em todo o conjunto de teste, e compare a distribuição dos erros das imagens do dígito 1 com a dos demais dígitos. Escolha um limiar a partir dessas distribuições e conte quantas imagens de cada dígito seriam sinalizadas como anômalas. Que dígitos são mais confundidos com o 1, e por quê?

In [ ]:
# Seu código aqui